In [ ]:
import pandas as pd
import plotly.express as px






In [8]:
import plotly.graph_objects as go
import pandas as pd

df_stars = pd.read_csv('../temp/stars.csv')


# Assuming df_stars is your dataframe
# df_stars = pd.DataFrame({'myindex': ..., 'crash': ..., 'boom': ...})

# Ensure the dataframe is sorted by myindex to prevent "random jumps"
df_plot = df_stars.sort_values('myindex')

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df_plot['crash'],
    y=df_plot['boom'],
    mode='lines+markers',
    name='Pareto Path',
    marker=dict(
        size=10,
        color=df_plot['myindex'], # Color code by index for visual flow
        colorscale='Viridis',
        showscale=True,
        colorbar=dict(title="Index")
    ),
    line=dict(width=2, color='rgba(100, 100, 100, 0.5)'),
    # Hover configuration
    customdata=df_plot[['myindex', 'crash', 'boom']],
    hovertemplate=(
        "<b>Index:</b> %{customdata[0]}<br>" +
        "<b>Crash:</b> %{customdata[1]:.4f}<br>" +
        "<b>Boom:</b> %{customdata[2]:.4f}<br>" +
        "<extra></extra>" # Removes the secondary trace name box
    )
))

fig.update_layout(
    title="Pareto Path: Crash vs Boom Objectives",
    xaxis_title="Crash Performance",
    yaxis_title="Boom Performance",
    hovermode="closest",
    template="plotly_white",
    width=900,
    height=600
)

fig.show()

In [2]:
import awswrangler as wr
import pandas as pd
database_name = table_name = "perturbation_evaluations_nav_1"
    # Read the table metadata and data using the Glue catalo

df_pert = wr.s3.read_parquet_table(
    table=table_name,
    database=database_name
)
PERTURBATION_FOLDER = "s3://jdinvestment/perturbations_nav_1"
df_parents = pd.read_csv("{}/perturbations.csv".format(PERTURBATION_FOLDER))
df_pert = df_pert.set_index('sim_id').join(df_parents[['sim_id', 'parent_id']].set_index('sim_id'))

df_gen0 = df_pert.loc[df_pert.boom > .15]
df_gen0.to_csv('../temp/generation_0_pert_nav.csv', index = False)

df_pert = df_pert[['parent_id', 'boom', 'crash']].groupby('parent_id').agg('mean')
df_pert = df_pert.sort_values(by = 'crash')


In [9]:
import numpy as np
import pandas as pd
from scipy.spatial.distance import cdist

import numpy as np
import pandas as pd
from scipy.spatial.distance import cdist
from sklearn.preprocessing import StandardScaler

def generate_diverse_seed_df(df, param_cols, obj_cols, senses, pop_size, top_k_percent=0.2):
    """
    Returns a DataFrame subset containing the most diverse individuals 
    selected from the top-performing tier.
    """
    # 1. Unified Ranking (Handles mixed 'min' and 'max' objectives)
    temp_df = df.copy()
    rank_scores = []
    
    for col, sense in zip(obj_cols, senses):
        if sense == 'max':
            rank_scores.append(temp_df[col].rank(ascending=False))
        else:
            rank_scores.append(temp_df[col].rank(ascending=True))
            
    # Calculate average rank across all objectives
    temp_df['_internal_avg_rank'] = pd.concat(rank_scores, axis=1).mean(axis=1)
    
    # 2. Filter for the "High Performance" candidate pool
    cutoff = int(len(df) * top_k_percent)
    # Ensure pool is at least the size of the requested population
    candidates = temp_df.nsmallest(max(cutoff, pop_size), '_internal_avg_rank').copy()
    
    # 3. Scale Parameters for Distance Calculation
    # This prevents parameters with large units from dominating the diversity metric
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(candidates[param_cols].values)
    
    # 4. Max-Min Diversity Selection (Furthest-First Traversal)
    selected_indices = [0]  # Seed with the #1 ranked performer
    remaining_indices = list(range(1, len(X_scaled)))
    
    while len(selected_indices) < pop_size and remaining_indices:
        # Distance from all remaining points to the current selected set
        dist_matrix = cdist(X_scaled[remaining_indices], X_scaled[selected_indices], metric='euclidean')
        
        # For each remaining point, find distance to its NEAREST already-selected neighbor
        min_dists = dist_matrix.min(axis=1)
        
        # Select the candidate whose "nearest neighbor" is the furthest away
        furthest_point_idx = np.argmax(min_dists)
        
        selected_indices.append(remaining_indices.pop(furthest_point_idx))
    
    # Return the clean subset without the internal ranking column
    return candidates.iloc[selected_indices].drop(columns=['_internal_avg_rank'])



param_cols = ["p_{}".format(i) for i in range(17)]
obj_cols = ['crash', 'boom']
senses = ['max', 'max']
pop_size = 180
df_pop = generate_diverse_seed_df(df_gen0, param_cols, obj_cols, senses, pop_size, top_k_percent=0.2)
df_pop.to_csv('../temp/robust_mean_starting_pop.csv', index = False)

In [8]:
import os
os.listdir('../temp')


['stars1.csv',
 'generation_0_pert_nav.csv',
 'stars.csv',
 'all_pareto_nav_stars.csv',
 'single_star.csv']

In [29]:
for col in param_cols:
    print(col, abs(df_pop[col].std()/df_pop[col].mean()))

p_0 0.01819747347485207
p_1 0.013738654820380466
p_2 0.03154301502480778
p_3 0.027426571033295294
p_4 0.01477535598481252
p_5 0.010292873599654391
p_6 0.010309768201651708
p_7 0.011579861256803197
p_8 1.507193833131567
p_9 0.8820062448291798
p_10 0.34396362144110215
p_11 0.6934312394910324
p_12 0.009868842153691893
p_13 0.009816289255118078
p_14 0.024827164015982776
p_15 0.10905581893208127
p_16 0.2529795145467781


In [4]:
import plotly.graph_objects as go
import pandas as pd

# Load both datasets
df_stars = pd.read_csv('../temp/stars.csv')
df_stars1 = pd.read_csv('../temp/median_nav_pareto_0.csv')

# Ensure both dataframes are sorted for a clean visual flow
df_plot = df_stars.sort_values('myindex')
df_plot1 = df_stars1

fig = go.Figure()

# Trace 1: stars.csv
fig.add_trace(go.Scatter(
    x=df_plot['crash'],
    y=df_plot['boom'],
    mode='lines+markers',
    name='Pareto Path (Stars)',
    marker=dict(
        size=10,
        color=df_plot['myindex'],
        colorscale='Viridis',
        showscale=True,
        colorbar=dict(title="Index", x=1.0)
    ),
    line=dict(width=2, color='rgba(100, 100, 100, 0.5)'),
    customdata=df_plot[['myindex', 'crash', 'boom']],
    hovertemplate=(
        "<b>Stars - Index:</b> %{customdata[0]}<br>" +
        "<b>Crash:</b> %{customdata[1]:.4f}<br>" +
        "<b>Boom:</b> %{customdata[2]:.4f}<br>" +
        "<extra></extra>"
    )
))

# Trace 2: stars1.csv
fig.add_trace(go.Scatter(
    x=df_plot1['crash'],
    y=df_plot1['boom'],
    mode='lines+markers',
    name='Pareto Path (Stars1)',
    marker=dict(
        size=10,
        #color=df_plot1['myindex'],
        colorscale='Plasma', # Different scale to distinguish datasets
        showscale=True,
        colorbar=dict(title="Index (Stars1)", x=1.15) # Offset second colorbar
    ),
    line=dict(width=2, dash='dash', color='rgba(200, 50, 50, 0.5)'),
    customdata=df_plot1[['crash', 'boom']],
    hovertemplate=(
        "<b>Stars1 - Index:</b> %{customdata[0]}<br>" +
        "<b>Crash:</b> %{customdata[1]:.4f}<br>" +
        "<b>Boom:</b> %{customdata[2]:.4f}<br>" +
        "<extra></extra>"
    )
))


# fig.add_trace(go.Scatter(
#     x=df_pert['crash'],
#     y=df_pert['boom'],
#     mode='lines+markers',
#     name='perturbattions',
#     marker=dict(
#         size=10,
#         color=df_plot1['myindex'],
#         colorscale='Plasma', # Different scale to distinguish datasets
#         showscale=True,
#         colorbar=dict(title="Index (Stars1)", x=1.15) # Offset second colorbar
#     ),
#     line=dict(width=2, dash='dash', color='rgba(200, 50, 50, 0.5)'),
#     customdata=df_pert[['crash', 'boom']],
#     hovertemplate=(
#         "<b>Stars1 - Index:</b> %{customdata[0]}<br>" +
#         "<b>Crash:</b> %{customdata[0]:.4f}<br>" +
#         "<b>Boom:</b> %{customdata[1]:.4f}<br>" +
#         "<extra></extra>"
#     )
# ))

fig.update_layout(
    title="Pareto Path Comparison: Crash vs Boom Objectives",
    xaxis_title="Crash Performance",
    yaxis_title="Boom Performance",
    hovermode="closest",
    template="plotly_white",
    width=1000,
    height=600,
    #legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01)
)

fig.show()

In [6]:
df_stars1.iloc[0].transpose()

boom       0.239944
crash      0.111896
myindex    0.000000
Name: 0, dtype: float64